## tl;dr

The school-year bridge has 31,336 unique nonempty `(_panel_year, 개방ID)` keys. It preserves 30,556 matched 0101 keys and all 780 unmatched keys. A left join from all 180,119,183 source rows has zero row expansion.

## Context & Methods

This validation notebook independently reloads the tracked bridge CSV, its build summary, and the earlier orphan-key list. It checks the bridge grain, status counts, orphan-key reconciliation, and per-dataset left-join cardinality. The full streaming rebuild and input checksum verification live in `scripts/build_edss_school_year_bridge.py`.

### Key Assumptions

- The intended bridge grain is one row per distinct nonempty `(_panel_year, 개방ID)` observed in any cataloged panel.
- Blank join keys remain in source fact rows and do not become a synthetic bridge member.
- 0101 numeric measures are not valid for implicit aggregation across campuses and therefore are excluded.

## Data

In [1]:
import csv
import json
import sys
from collections import Counter
from pathlib import Path

repo_root = Path.cwd()
if not (repo_root / 'data' / 'metadata').exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / 'scripts'))
import build_edss_school_year_bridge as bridge_builder

bridge_path = repo_root / 'data' / 'metadata' / 'edss_school_year_bridge.csv'
summary_path = repo_root / 'data' / 'metadata' / 'edss_school_year_bridge_summary.json'
orphan_path = repo_root / 'data' / 'metadata' / 'edss_full_panel_orphan_school_year_keys.csv'

with bridge_path.open(encoding='utf-8-sig', newline='') as handle:
    bridge_rows = list(csv.DictReader(handle))
summary = json.loads(summary_path.read_text(encoding='utf-8'))
with orphan_path.open(encoding='utf-8-sig', newline='') as handle:
    orphan_rows = list(csv.DictReader(handle))

{'bridge': str(bridge_path.relative_to(repo_root)), 'rows': len(bridge_rows), 'summary': str(summary_path.relative_to(repo_root))}

{'bridge': 'data/metadata/edss_school_year_bridge.csv',
 'rows': 31336,
 'summary': 'data/metadata/edss_school_year_bridge_summary.json'}

## Results

### 1. Validate the bridge grain

In [2]:
grain_validation = bridge_builder.validate_bridge_records(bridge_rows)
assert grain_validation['row_count'] == 31_336
assert grain_validation['distinct_key_count'] == 31_336
assert grain_validation['duplicate_key_count'] == 0
assert grain_validation['max_key_multiplicity'] == 1
grain_validation

{'key_columns': ['_panel_year', '개방ID'],
 'row_count': 31336,
 'distinct_key_count': 31336,
 'duplicate_key_count': 0,
 'blank_key_count': 0,
 'max_key_multiplicity': 1,
 'unique_key': True}

### 2. Reconcile match and review statuses

In [3]:
match_counts = Counter(row['_0101_match_status'] for row in bridge_rows)
review_counts = Counter(row['_review_status'] for row in bridge_rows)
expected_match_counts = {
    'matched': 30_556,
    'before_base_first_seen': 63,
    'after_base_last_seen': 626,
    'internal_base_gap': 49,
    'open_id_absent_all_years': 42,
}
assert dict(match_counts) == expected_match_counts
assert review_counts['external_crosscheck_required_internal_gap'] == 49
assert review_counts['external_crosscheck_required_absent_all_years'] == 42
{'match_status_counts': dict(match_counts), 'review_status_counts': dict(review_counts)}

{'match_status_counts': {'matched': 30556,
  'open_id_absent_all_years': 42,
  'before_base_first_seen': 63,
  'internal_base_gap': 49,
  'after_base_last_seen': 626},
 'review_status_counts': {'not_required': 30556,
  'external_crosscheck_required_absent_all_years': 42,
  'unresolved_temporal_boundary': 689,
  'external_crosscheck_required_internal_gap': 49}}

### 3. Match the full-panel orphan-key audit exactly

In [4]:
bridge_orphans = {
    (row['_panel_year'], row['개방ID']): row['_0101_match_status']
    for row in bridge_rows
    if row['_0101_exists'] == 'false'
}
classification_map = {
    'before_first_0101_year': 'before_base_first_seen',
    'after_last_0101_year': 'after_base_last_seen',
    'internal_0101_gap': 'internal_base_gap',
    'never_in_0101': 'open_id_absent_all_years',
}
diagnosed_orphans = {
    (row['year'], row['open_id']): classification_map[row['classification']]
    for row in orphan_rows
}
assert bridge_orphans == diagnosed_orphans
{'bridge_orphan_keys': len(bridge_orphans), 'diagnosed_orphan_keys': len(diagnosed_orphans), 'exact_match': True}

{'bridge_orphan_keys': 780, 'diagnosed_orphan_keys': 780, 'exact_match': True}

### 4. Confirm no row expansion for every dataset

In [5]:
dataset_checks = summary['datasets']
assert len(dataset_checks) == 233
assert all(row['left_join_output_row_count'] == row['input_row_count'] for row in dataset_checks)
assert all(row['row_expansion_count'] == 0 for row in dataset_checks)
assert summary['source_input_row_count'] == 180_119_183
assert summary['source_unmatched_0101_row_count'] == 82_959
assert summary['input_checksum_validation'] == {'status': 'pass', 'checked_file_count': 233, 'mismatch_count': 0}
{
    'datasets_checked': len(dataset_checks),
    'source_rows': summary['source_input_row_count'],
    'left_join_rows': summary['left_join_validation']['simulated_left_join_row_count'],
    'row_expansion': summary['left_join_validation']['row_expansion_count'],
    'missing_join_key_rows_preserved': summary['source_missing_join_key_row_count'],
}

{'datasets_checked': 233,
 'source_rows': 180119183,
 'left_join_rows': 180119183,
 'row_expansion': 0,
 'missing_join_key_rows_preserved': 46962}

## Takeaways

- The bridge is safe for many-to-one left joins at `(_panel_year, 개방ID)` and does not expand source rows.
- All 780 unmatched school-year keys remain explicit review cases; no adjacent-year mapping was applied.
- The 46,962 rows with missing join keys remain source-side rows and are not collapsed into a fake school.
- Campus lists describe categorical coverage only. Any campus-level numeric analysis must return to raw 0101 grain.